# POSEIDON Retrieval — Uncontaminated (100 transits)

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd

from POSEIDON.constants import M_E, R_E, R_Sun
from POSEIDON.core import (
    create_planet,
    create_star,
    define_model,
    load_data,
    read_opacities,
    set_priors,
    wl_grid_constant_R,
)
from POSEIDON.corner import generate_cornerplot
from POSEIDON.instrument import bin_spectrum_to_data
from POSEIDON.utility import read_data, read_retrieved_spectrum, plot_collection
from POSEIDON.visuals import plot_data, plot_spectra_retrieved


# -----------------------------
# Case configuration (edit as needed)
# -----------------------------
PLANET_NAME = "Trappist-1e"
OBSERVATION_FILE = "pandexo_output_10transits_fspot0.00_ffac0.00.dat"
DATA_DIR = Path("observations")
INSTRUMENTS = ["JWST_NIRSpec_PRISM"]

MODEL_NAME = "uncontam_10T_0.00spot-0.00fac"
BULK_SPECIES = ["N2"]
PARAM_SPECIES = ["H2O", "CH4", "CO2", "O3"]

CLEAN_PATH = Path("pandexo_spec.txt")

# Wavelength grid for model initialization
WL_MIN_UM = 0.4
WL_MAX_UM = 6.0
R_GRID = 4000


## 1) Star and planet definitions


In [ ]:
# -----------------------------
# Star definition (TRAPPIST-1)
# -----------------------------
R_s = 0.1192 * R_Sun
T_s = 2566.0
met_s = 0.00
log_g_s = 5.2396

star = create_star(
    R_s,
    T_s,
    log_g_s,
    met_s,
    stellar_grid="phoenix",
)


# -----------------------------
# Planet definition (TRAPPIST-1e)
# -----------------------------
R_p = 0.917985 * R_E
M_p = 0.6356 * M_E
T_eq = 255.0

planet = create_planet(PLANET_NAME, R_p, mass=M_p, T_eq=T_eq)


## 2) Load dataset and visualize


In [ ]:
wl_model = wl_grid_constant_R(WL_MIN_UM, WL_MAX_UM, R_GRID)

data = load_data(
    str(DATA_DIR),
    datasets=[OBSERVATION_FILE],
    instruments=INSTRUMENTS,
    wl=wl_model,
)

fig_data = plot_data(data, PLANET_NAME)


## 3) Define retrieval model and priors


In [ ]:
model = define_model(
    MODEL_NAME,
    BULK_SPECIES,
    PARAM_SPECIES,
    PT_profile="isotherm",
    cloud_model="cloud-free",
)

print("Free parameters:", model["param_names"])


prior_types = {
    "T": "uniform",
    "R_p_ref": "uniform",
    "log_H2O": "uniform",
    "log_CH4": "uniform",
    "log_CO2": "uniform",
    "log_O3": "uniform",
}

prior_ranges = {
    "T": [200, 400],
    "R_p_ref": [0.85 * R_p, 1.15 * R_p],
    "log_H2O": [-8, -1],
    "log_CH4": [-8, -1],
    "log_CO2": [-5, -1],
    "log_O3": [-8, -1],
}

priors = set_priors(planet, star, model, data, prior_types, prior_ranges)


## 4) Read and pre-interpolate opacities


In [ ]:
opacity_treatment = "opacity_sampling"

# Temperature grid (K)
T_fine = np.arange(200, 401, 10)

# Pressure grid in log10(P/bar)
log_P_fine = np.arange(-2, 2.0001, 0.2)

opac = read_opacities(model, wl_model, opacity_treatment, T_fine, log_P_fine)


## 5) Plot retrieved spectrum and corner plot

In [ ]:
(
    wl_out,
    spec_low2,
    spec_low1,
    spec_median,
    spec_high1,
    spec_high2,
) = read_retrieved_spectrum(PLANET_NAME, MODEL_NAME)

spectra_median = plot_collection(spec_median, wl_out, collection=[])
spectra_low1 = plot_collection(spec_low1, wl_out, collection=[])
spectra_low2 = plot_collection(spec_low2, wl_out, collection=[])
spectra_high1 = plot_collection(spec_high1, wl_out, collection=[])
spectra_high2 = plot_collection(spec_high2, wl_out, collection=[])

fig_spec = plot_spectra_retrieved(
    spectra_median,
    spectra_low2,
    spectra_low1,
    spectra_high1,
    spectra_high2,
    PLANET_NAME,
    data,
    R_to_bin=100,
    data_labels=["NIRSpec_PRISM"],
    data_colour_list=["lime"],
)

fig_corner = generate_cornerplot(planet, model)


## 6) Clean-truth rebinned metrics (MSE / reduced $\chi^2$)

In [ ]:
N_FREE_PARAMS = 11
LOG_PATH = Path("chi2_log_finale.csv")


def load_clean_two_cols(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """Load a two-column clean spectrum file: (wl_um, depth)."""
    arr = np.genfromtxt(path, comments="#", dtype=float)
    if arr.ndim == 1:
        arr = arr.reshape(-1, 1)
    if arr.shape[1] < 2:
        raise ValueError(
            "Expected at least 2 columns (wl_um, depth) in the clean file."
        )
    wl_clean = arr[:, 0].astype(float)
    y_clean = arr[:, 1].astype(float)

    idx = np.argsort(wl_clean)
    return wl_clean[idx], y_clean[idx]


def bin_average_with_halfbins(
    wl_src: np.ndarray,
    y_src: np.ndarray,
    centers: np.ndarray,
    halfwidths: np.ndarray,
    nsamp: int = 256,
) -> np.ndarray:
    """Band-average y_src over [c-h, c+h] using trapezoidal integration."""
    wl_src = np.asarray(wl_src, dtype=float)
    y_src = np.asarray(y_src, dtype=float)
    centers = np.asarray(centers, dtype=float)
    halfwidths = np.asarray(halfwidths, dtype=float)

    sort_idx = np.argsort(wl_src)
    wl_sorted = wl_src[sort_idx]
    y_sorted = y_src[sort_idx]

    out = np.empty_like(centers, dtype=float)
    for i, (c, h) in enumerate(zip(centers, halfwidths)):
        a, b = c - h, c + h
        x = np.linspace(a, b, nsamp)
        yx = np.interp(x, wl_sorted, y_sorted)
        out[i] = np.trapz(yx, x) / (b - a)
    return out


# Load clean truth and observed dataset binning
wl_clean, y_clean = load_clean_two_cols(CLEAN_PATH)
wl_data, half_bin, y_obs, err_obs = read_data(str(DATA_DIR), OBSERVATION_FILE)

print(f"N observed points: {len(wl_data)}")
print(f"wl_data range: {wl_data.min():.4f}–{wl_data.max():.4f} um")

# Rebin retrieved median spectrum to dataset bins
model_binned = bin_spectrum_to_data(spec_median, wl_out, data)

# Rebin clean truth to dataset bins
clean_binned = bin_average_with_halfbins(wl_clean, y_clean, wl_data, half_bin)

if not (len(model_binned) == len(clean_binned) == len(err_obs)):
    raise ValueError("Binned arrays and errors must have the same length.")

sig = np.asarray(err_obs, dtype=float)
if np.any(sig <= 0):
    raise ValueError("Found non-positive uncertainties in err_obs; chi2 is undefined.")

resid = np.asarray(model_binned, dtype=float) - np.asarray(clean_binned, dtype=float)

N = int(resid.size)
p = int(N_FREE_PARAMS)
dof = int(max(N - p, 0))

chi2 = float(np.sum((resid / sig) ** 2))
chi2_reduced = chi2 / dof if dof > 0 else np.nan
mse = float(np.mean(resid**2))
rmse = float(np.sqrt(mse))

print("---- Metrics vs clean truth (no mask) ----")
print(f"N points      : {N}")
print(f"Free params p : {p}")
print(f"DoF           : {dof}")
print(f"MSE           : {mse:.6e}")
print(f"RMSE          : {rmse:.6e} ({1e6 * rmse:.2f} ppm)")
print(f"chi2          : {chi2:.6f}")
print(f"chi2_reduced  : {chi2_reduced:.6f}")

row = {
    "planet_name": PLANET_NAME,
    "model_name": MODEL_NAME,
    "observation": OBSERVATION_FILE,
    "N": N,
    "p": p,
    "dof": dof,
    "MSE": mse,
    "chi2": chi2,
    "chi2_reduced": chi2_reduced,
}

if LOG_PATH.exists():
    df_log = pd.read_csv(LOG_PATH)
    df_log = pd.concat([df_log, pd.DataFrame([row])], ignore_index=True)
else:
    df_log = pd.DataFrame([row])

df_log.to_csv(LOG_PATH, index=False, float_format="%.10g")
print(f"Appended row to: {LOG_PATH.resolve()}")
